In [5]:
import pandas as pd
import numpy as np

# -------------------------------
# 1. WITH LIBRARIES (sklearn)
# -------------------------------
def linreg_with_libs(csv_path):
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    df = pd.read_csv(csv_path)

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    return float(mae), float(mse), float(rmse)


# -------------------------------
# 2. WITHOUT LIBRARIES (Manual)
# -------------------------------
def linreg_from_scratch(csv_path):
    df = pd.read_csv(csv_path)

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values.reshape(-1, 1)

    # add bias column
    X = np.hstack((np.ones((X.shape[0], 1)), X))

    # 80/20 split
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # normal equation
    theta = np.linalg.pinv(X_train.T @ X_train) @ X_train.T @ y_train

    y_pred = X_test @ theta

    mae = np.mean(np.abs(y_test - y_pred))
    mse = np.mean((y_test - y_pred) ** 2)
    rmse = np.sqrt(mse)

    return float(mae), float(mse), float(rmse)


# -------------------------------
# RUN BOTH
# -------------------------------
path = "/content/canada_per_capita_income.csv"

print("With libraries (MAE, MSE, RMSE):", linreg_with_libs(path))
print("From scratch (MAE, MSE, RMSE):", linreg_from_scratch(path))

With libraries (MAE, MSE, RMSE): (3240.91399747583, 15147815.5477862, 3892.0194690913613)
From scratch (MAE, MSE, RMSE): (10105.271821692431, 116383628.51629825, 10788.124420690478)


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load data
df = pd.read_csv("hiring.csv")

# Data Preprocessing
# Convert 'experience' column from words to numbers
word_to_num = {
    'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11
}
df['experience'] = df['experience'].replace(word_to_num).infer_objects(copy=False)

# Drop rows with NaN values in 'experience' or 'test_score(out of 10)'
df = df.dropna(subset=['experience', 'test_score(out of 10)'])

# Split features and target (last column is target)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------- MANUAL MODEL ----------------
def manual_regression(X_train, y_train, X_test):
    # add intercept
    X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
    X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

    # normal equation
    theta = np.linalg.pinv(X_train_b.T.dot(X_train_b)).dot(
        X_train_b.T
    ).dot(y_train)

    return X_test_b.dot(theta)

y_pred_manual = manual_regression(X_train, y_train, X_test)

# ---------------- SKLEARN MODEL ----------------
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_sklearn = model.predict(X_test)

# ---------------- METRICS FUNCTION ----------------
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return mse, rmse, mae

# Evaluate manual
mse_m, rmse_m, mae_m = metrics(y_test, y_pred_manual)

# Evaluate sklearn
mse_s, rmse_s, mae_s = metrics(y_test, y_pred_sklearn)

# ---------------- OUTPUT ----------------
print("MANUAL MODEL")
print("MSE :", mse_m)
print("RMSE:", rmse_m)
print("MAE :", mae_m)

print("\nSKLEARN MODEL")
print("MSE :", mse_s)
print("RMSE:", rmse_s)
print("MAE :", mae_s)

MANUAL MODEL
MSE : 472656.24999829923
RMSE: 687.4999999987631
MAE : 687.4999999987631

SKLEARN MODEL
MSE : 472656.25
RMSE: 687.5
MAE : 687.5


/tmp/ipykernel_3143/923960533.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['experience'] = df['experience'].replace(word_to_num).infer_objects(copy=False)
